In [1]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv('../Dataset/Raw/support_tickets_raw.csv')

# --- Initial inspection ---
print(df.shape)
print(df.info())
print(df.isnull().sum())
print(df.describe(include='all').T)
df.head()

(100000, 20)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ticket_id              100000 non-null  str    
 1   created_at             100000 non-null  str    
 2   customer_id            100000 non-null  str    
 3   customer_segment       100000 non-null  str    
 4   channel                100000 non-null  str    
 5   product_area           100000 non-null  str    
 6   issue_type             100000 non-null  str    
 7   priority               100000 non-null  str    
 8   status                 100000 non-null  str    
 9   sla_plan               100000 non-null  str    
 10  initial_message        100000 non-null  str    
 11  agent_first_reply      100000 non-null  str    
 12  resolution_summary     60113 non-null   str    
 13  resolution_time_hours  60113 non-null   float64
 14  reopened               100000 non-n

,ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region
0,TCKT_000001,2024-01-31T05:14:27,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0,very_negative,1,0,android,EU
1,TCKT_000002,2024-10-20T06:15:49,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is r...,Ticket closed without further action after no ...,238.32,0,neutral,3,0,web,NaN
2,TCKT_000003,2024-06-18T21:35:54,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my c...,Thanks for reporting this bug. We will look in...,NaN,NaN,0,neutral,3,0,android,MEA
3,TCKT_000004,2025-12-25T15:59:52,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,NaN,NaN,0,positive,5,1,android,LATAM
4,TCKT_000005,2023-08-27T16:08:33,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,61.32,0,very_negative,2,0,web,NaN


In [2]:
print(df.shape)
print("---DTYPES---")
print(df.dtypes)
print("---NULLS---")
print(df.isnull().sum())
print("---UNIQUE VALUE COUNTS---")
for col in ['priority', 'status', 'sla_plan', 'customer_sentiment', 'channel', 
            'product_area', 'issue_type', 'customer_segment', 'region', 
            'platform', 'reopened', 'has_attachment']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

(100000, 20)
---DTYPES---
ticket_id                    str
created_at                   str
customer_id                  str
customer_segment             str
channel                      str
product_area                 str
issue_type                   str
priority                     str
status                       str
sla_plan                     str
initial_message              str
agent_first_reply            str
resolution_summary           str
resolution_time_hours    float64
reopened                   int64
customer_sentiment           str
csat_score                 int64
has_attachment             int64
platform                     str
region                       str
dtype: object
---NULLS---
ticket_id                    0
created_at                   0
customer_id                  0
customer_segment             0
channel                      0
product_area                 0
issue_type                   0
priority                     0
status                       0
sla_plan 

In [3]:
# --- 1. Datetime conversion ---
df['created_at'] = pd.to_datetime(df['created_at'])
df['created_month'] = df['created_at'].dt.to_period('M').astype(str)
df['created_dow'] = df['created_at'].dt.day_name()
df['created_hour'] = df['created_at'].dt.hour

# --- 2. Structural null handling (do NOT impute — these are legitimate) ---
df['is_terminal'] = df['status'].isin(['resolved', 'closed_no_action'])

# --- 3. Region: true missing data ---
df['region'] = df['region'].fillna('Unknown')

# --- 4. Define SLA target matrix (documented business assumption) ---
sla_matrix = {
    ('urgent', 'standard'): 8,  ('urgent', 'gold'): 4,  ('urgent', 'platinum'): 2,
    ('high',   'standard'): 24, ('high',   'gold'): 12, ('high',   'platinum'): 6,
    ('medium', 'standard'): 48, ('medium', 'gold'): 24, ('medium', 'platinum'): 12,
    ('low',    'standard'): 72, ('low',    'gold'): 48, ('low',    'platinum'): 24,
}
df['sla_target_hours'] = df.apply(
    lambda r: sla_matrix[(r['priority'], r['sla_plan'])], axis=1
)

# --- 5. SLA breach flag — only defined for terminal tickets ---
df['sla_breached'] = np.where(
    df['is_terminal'],
    df['resolution_time_hours'] > df['sla_target_hours'],
    np.nan  # undefined for tickets still open
)

# --- 6. Ordinal sentiment score (useful for correlation/modeling later) ---
sentiment_map = {
    'very_negative': -2, 'negative': -1, 'neutral': 0,
    'positive': 1, 'very_positive': 2
}
df['sentiment_score'] = df['customer_sentiment'].map(sentiment_map)

# --- 7. Memory-efficient dtypes ---
cat_cols = ['customer_segment', 'channel', 'product_area', 'issue_type', 'priority',
            'status', 'sla_plan', 'customer_sentiment', 'platform', 'region']
for col in cat_cols:
    df[col] = df[col].astype('category')

# --- 8. Sanity checks before saving ---
print(df['sla_breached'].value_counts(dropna=False))
print(df.groupby('priority')['sla_target_hours'].unique())
print(df.shape)
df.info()

# --- 9. Save cleaned dataset ---
df.to_csv('../Dataset/Cleaned/support_tickets_cleaned.csv', index=False)

sla_breached
0.0    40125
NaN    39887
1.0    19988
Name: count, dtype: int64
priority
high       [6, 24, 12]
low       [72, 48, 24]
medium    [48, 24, 12]
urgent       [4, 8, 2]
Name: sla_target_hours, dtype: object
(100000, 27)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ticket_id              100000 non-null  str           
 1   created_at             100000 non-null  datetime64[us]
 2   customer_id            100000 non-null  str           
 3   customer_segment       100000 non-null  category      
 4   channel                100000 non-null  category      
 5   product_area           100000 non-null  category      
 6   issue_type             100000 non-null  category      
 7   priority               100000 non-null  category      
 8   status                 100000 non-null  category      
 9   sla_pl

In [4]:
check = pd.read_csv('../Dataset/Cleaned/support_tickets_cleaned.csv')
print(check.shape)
print(check['sla_breached'].value_counts(dropna=False))

(100000, 27)
sla_breached
0.0    40125
NaN    39887
1.0    19988
Name: count, dtype: int64
